In [25]:
using LinearAlgebra
using Printf


In [30]:
abstract type FiniteGroup end
abstract type FiniteGroupElement end 
abstract type FiniteGroupRepresentation end


tetrahedral_symbol = [:E, :C2x, :C2y, :C2z, :C3a, :C3a2, :C3b, :C3b2, :C3c, :C3c2, :C3d, :C3d2]

struct TetrahedralElement <: FiniteGroupElement   
    sym::Symbol
    rep::Matrix{Int64}
    function char2tet(c::Char)
        if c== 'a'
            return [1 0 0 0]
        elseif c== 'b'
            return [0 1 0 0]
        elseif c== 'c' 
            return [0 0 1 0]
        elseif c== 'd'
            return [0 0 0 1]
        else
            throw(ArgumentError("Invalid character: $c"))
        end
    end
    function TetrahedralElement(s::Symbol, nn::String)
        @assert length(nn) == 4
        @assert all(c -> c in "abcd", nn)
        mat = zeros(Int64, 4, 4) 
        for i in 1:4
            mat[:, i] = char2tet(nn[i])
        end
        return new(s, mat)
    end
end

struct Tetrahedral<:FiniteGroup
    elements::Dict{Symbol, TetrahedralElement}
    function Tetrahedral()
        els = Dict(
        :E => TetrahedralElement(:E, "abcd"),
        :C2x => TetrahedralElement(:C2x, "badc"),
        :C2y => TetrahedralElement(:C2y, "dcba"), 
        :C2z => TetrahedralElement(:C2z, "cdab"), 
        :C3a => TetrahedralElement(:C3a, "adbc"),
        :C3a2 => TetrahedralElement(:C3a2, "acdb"),
        :C3b => TetrahedralElement(:C3b, "cbda"),
        :C3b2 => TetrahedralElement(:C3b2, "dbac"),
        :C3c => TetrahedralElement(:C3c, "dacb"),
        :C3c2 => TetrahedralElement(:C3c2, "bdca"),
        :C3d => TetrahedralElement(:C3d, "bcad"),
        :C3d2 => TetrahedralElement(:C3d2, "cabd"))
        return new(els)
    end

end

function (p::Tetrahedral)(s::Symbol)
    @assert s in [:E, :C2x, :C2y, :C2z, :C3a, :C3a2, :C3b, :C3b2, :C3c, :C3c2, :C3d, :C3d2]
    return p.elements[s]
end

function find_in_Group(G::T, rep) where T<:FiniteGroup
    for (key, value) in G.elements
        if value.rep == rep
            return G(key)
        end
    end
    return nothing
end


Base.show(io::IO, t::TetrahedralElement) = print(io, "$(t.sym) [TetrahedralElement] : ", t.rep)
# Base.*(a::TetrahedralElement, b::TetrahedralElement) = find_in_Group(Tetrahedral(), a.rep * b.rep)

    

In [31]:
E=TetrahedralElement(:E, "abcd")

E [TetrahedralElement] : [1 0 0 0; 0 1 0 0; 0 0 1 0; 0 0 0 1]

In [32]:
Tet=Tetrahedral()

Tetrahedral(Dict{Symbol, TetrahedralElement}(:C3c => C3c [TetrahedralElement] : [0 1 0 0; 0 0 0 1; 0 0 1 0; 1 0 0 0], :C3b => C3b [TetrahedralElement] : [0 0 0 1; 0 1 0 0; 1 0 0 0; 0 0 1 0], :C3d => C3d [TetrahedralElement] : [0 0 1 0; 1 0 0 0; 0 1 0 0; 0 0 0 1], :C3a2 => C3a2 [TetrahedralElement] : [1 0 0 0; 0 0 0 1; 0 1 0 0; 0 0 1 0], :C2x => C2x [TetrahedralElement] : [0 1 0 0; 1 0 0 0; 0 0 0 1; 0 0 1 0], :C3d2 => C3d2 [TetrahedralElement] : [0 1 0 0; 0 0 1 0; 1 0 0 0; 0 0 0 1], :C3b2 => C3b2 [TetrahedralElement] : [0 0 1 0; 0 1 0 0; 0 0 0 1; 1 0 0 0], :C3c2 => C3c2 [TetrahedralElement] : [0 0 0 1; 1 0 0 0; 0 0 1 0; 0 1 0 0], :C3a => C3a [TetrahedralElement] : [1 0 0 0; 0 0 1 0; 0 0 0 1; 0 1 0 0], :E => E [TetrahedralElement] : [1 0 0 0; 0 1 0 0; 0 0 1 0; 0 0 0 1]…))

In [33]:
find_in_Group(Tet, Tet(:C2x).rep)

C2x [TetrahedralElement] : [0 1 0 0; 1 0 0 0; 0 0 0 1; 0 0 1 0]

In [54]:
for k1 in tetrahedral_symbol
    for (i, k2) in enumerate(tetrahedral_symbol)
        result = Tet(k1).rep * Tet(k2).rep
        found_element = find_in_Group(Tet, result)
        if found_element !== nothing
            q = String(found_element.sym)
            if i == 1
                @printf("%12s", "\$"*q*"\$ &")
            end
            
            if length(q) == 1
                @printf("%12s", "\$"*q*"\$ &")
            else
                @printf("%12s", "\$"*q[1]*"_{"*q[2:end]*"}\$ &")
            end
            
        else 
            @printf("%6s", "-------") 
        end
    end
    print("\\\\ \n")
end

       $E$ &       $E$ &  $C_{2x}$ &  $C_{2y}$ &  $C_{2z}$ &  $C_{3a}$ & $C_{3a2}$ &  $C_{3b}$ & $C_{3b2}$ &  $C_{3c}$ & $C_{3c2}$ &  $C_{3d}$ & $C_{3d2}$ &\\ 
     $C2x$ &  $C_{2x}$ &       $E$ &  $C_{2z}$ &  $C_{2y}$ &  $C_{3d}$ & $C_{3c2}$ &  $C_{3c}$ & $C_{3d2}$ &  $C_{3b}$ & $C_{3a2}$ &  $C_{3a}$ & $C_{3b2}$ &\\ 
     $C2y$ &  $C_{2y}$ &  $C_{2z}$ &       $E$ &  $C_{2x}$ &  $C_{3c}$ & $C_{3b2}$ &  $C_{3d}$ & $C_{3a2}$ &  $C_{3a}$ & $C_{3d2}$ &  $C_{3b}$ & $C_{3c2}$ &\\ 
     $C2z$ &  $C_{2z}$ &  $C_{2y}$ &  $C_{2x}$ &       $E$ &  $C_{3b}$ & $C_{3d2}$ &  $C_{3a}$ & $C_{3c2}$ &  $C_{3d}$ & $C_{3b2}$ &  $C_{3c}$ & $C_{3a2}$ &\\ 
     $C3a$ &  $C_{3a}$ &  $C_{3c}$ &  $C_{3b}$ &  $C_{3d}$ & $C_{3a2}$ &       $E$ & $C_{3c2}$ &  $C_{2z}$ & $C_{3d2}$ &  $C_{2y}$ & $C_{3b2}$ &  $C_{2x}$ &\\ 
    $C3a2$ & $C_{3a2}$ & $C_{3d2}$ & $C_{3c2}$ & $C_{3b2}$ &       $E$ &  $C_{3a}$ &  $C_{2y}$ &  $C_{3d}$ &  $C_{2x}$ &  $C_{3b}$ &  $C_{2z}$ &  $C_{3c}$ &\\ 
     $C3b$ &  $C_{3b}$ &  $C_{3d}$ &  $C

In [ ]:
Tet